# 09 — Full Pipeline: Orchestrator + All 5 Artifacts

⚙️ Hybrid

`harmonize_patient` is the entry point for the entire Layer 3 pipeline. It reads silver bundles, runs every sub-task in dependency order, and writes gold-tier output. This notebook re-runs the pipeline (idempotent), inspects the manifest, and locates each of the 5 showcase artifacts in the gold bundle.


## 1. Run harmonize_patient (idempotent)

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
CORPUS     = ATLAS_ROOT / "corpus"

from ehi_atlas.harmonize.orchestrator import harmonize_patient

result = harmonize_patient(
    silver_root = CORPUS / "silver",
    bronze_root = CORPUS / "bronze",
    gold_root   = CORPUS / "gold",
    patient_id  = "rhett759",
)

print("patient_id:   ", result.patient_id)
print("source_count: ", result.source_count)
print("conflicts:    ", result.conflict_count)
print("bundle_sha256:", result.bundle_sha256[:16], "...")


## 2. manifest.json

In [ ]:
manifest = json.loads(result.manifest_path.read_text())
print("Built at:", manifest["built_at"])
print("Sources:")
for s in manifest["sources"]:
    print(f"  {s['name']}")
print()
print("Resource counts:")
for rtype, count in sorted(manifest["resource_counts"].items()):
    print(f"  {rtype}: {count}")
print()
print("Merge summary:", manifest["merge_summary"])


## 3. Load the gold bundle

In [ ]:
gold_bundle = json.loads(result.bundle_path.read_text())
entries = gold_bundle["entry"]
print(f"Total gold resources: {len(entries)}")

from collections import Counter
types = Counter(e["resource"]["resourceType"] for e in entries)
for rtype, count in sorted(types.items()):
    print(f"  {rtype}: {count}")


## 4. Artifact 1 — Hyperlipidemia merge (SNOMED + ICD-10 → one Condition)

In [ ]:
# 🔧 Script + 📚 Reference
from ehi_atlas.harmonize.provenance import EXT_UMLS_CUI

gold_hyperlipid = None
for entry in gold_bundle["entry"]:
    res = entry["resource"]
    if res.get("resourceType") != "Condition":
        continue
    codes = {c.get("code") for c in res.get("code", {}).get("coding", [])}
    if codes & {"55822004", "E78.5"}:
        gold_hyperlipid = res
        break

if gold_hyperlipid:
    print("Artifact 1 — Condition:", gold_hyperlipid["id"])
    for coding in gold_hyperlipid["code"]["coding"]:
        cui = next((e["valueString"] for e in coding.get("extension", [])
                    if "umls-cui" in e.get("url","")), None)
        print(f"  {coding.get('system','').split('/')[-1]} {coding['code']} → CUI {cui}")
    src_tags = [t["code"] for t in gold_hyperlipid.get("meta",{}).get("tag",[])
                if "source-tag" in t.get("system","")]
    print("  source-tags:", src_tags)
else:
    print("Artifact 1 not found")


## 5. Artifact 2 — statin MedicationRequests with EXT_CONFLICT_PAIR

In [ ]:
# 🔧 Script + 📚 Reference
from ehi_atlas.harmonize.provenance import EXT_CONFLICT_PAIR

statin_meds = []
for entry in gold_bundle["entry"]:
    res = entry["resource"]
    if res.get("resourceType") != "MedicationRequest":
        continue
    rxcuis = {c.get("code") for c in res.get("medicationCodeableConcept",{}).get("coding",[])}
    if rxcuis & {"316672", "83367", "36567"}:
        statin_meds.append(res)

print(f"Artifact 2 — statin MedicationRequests: {len(statin_meds)}")
for med in statin_meds:
    rxcui = next((c.get("code") for c in med.get("medicationCodeableConcept",{}).get("coding",[])), "?")
    conflict = next((e.get("valueReference",{}).get("reference") for e in med.get("extension",[])
                     if e.get("url") == EXT_CONFLICT_PAIR), None)
    print(f"  {med['id']}  rxcui={rxcui}  status={med['status']}  conflict_pair → {conflict}")


## 6. Artifact 3 — single-source Claim (pass-through)

In [ ]:
# Claim resources are pass-through (no merge logic in Phase 1)
claims = [e["resource"] for e in gold_bundle["entry"]
          if e["resource"]["resourceType"] == "Claim"]
print(f"Artifact 3 — Claims in gold: {len(claims)}")
if claims:
    c0 = claims[0]
    src_tags = [t["code"] for t in c0.get("meta",{}).get("tag",[])
                if "source-tag" in t.get("system","")]
    print(f"  Example: {c0['id']}  source-tags: {src_tags}")
    print(f"  (Single source — pass-through; no cross-source merge)")


## 7. Artifact 4 — synthesized clinical note DocumentReference

In [ ]:
# Phase-1 partial: DocumentReference present; NLP extraction of chest-tightness
# Condition is a Phase-2 gap (requires vision wrapper wired to clinical note).
doc_refs = [e["resource"] for e in gold_bundle["entry"]
            if e["resource"]["resourceType"] == "DocumentReference"]
print(f"Artifact 4 — DocumentReferences in gold: {len(doc_refs)}")
for dr in doc_refs:
    src_tags = [t["code"] for t in dr.get("meta",{}).get("tag",[])
                if "source-tag" in t.get("system","")]
    loinc = next(
        (c.get("code") for c in dr.get("type",{}).get("coding",[])),
        "?"
    )
    print(f"  {dr['id']}  type-LOINC={loinc}  source={src_tags}")
    print(f"  Phase-2 gap: chest-tightness Condition extraction requires vision wrapper on .txt")


## 8. Artifact 5 — creatinine merged Observation (epic-ehi + lab-pdf)

In [ ]:
from ehi_atlas.harmonize.provenance import EXT_MERGE_RATIONALE

creatinine = None
for entry in gold_bundle["entry"]:
    res = entry["resource"]
    if res.get("resourceType") != "Observation":
        continue
    if any(c.get("code") == "2160-0" for c in res.get("code",{}).get("coding",[])):
        creatinine = res
        break

if creatinine:
    print("Artifact 5 — Observation:", creatinine["id"])
    print("  effectiveDateTime:", creatinine.get("effectiveDateTime"))
    print("  valueQuantity:", creatinine.get("valueQuantity"))
    src_tags = [t["code"] for t in creatinine.get("meta",{}).get("tag",[])
                if "source-tag" in t.get("system","")]
    print("  source-tags:", src_tags)
    rationale = next((e.get("valueString") for e in creatinine.get("meta",{}).get("extension",[])
                      if EXT_MERGE_RATIONALE in e.get("url","")), None)
    print("  rationale:", rationale)
else:
    print("Artifact 5 not found")


## 9. Provenance graph — walking a MERGE edge

In [ ]:
# Read provenance.ndjson and show one MERGE edge for a merged Condition
provenance_records = []
with result.provenance_path.open() as fh:
    for line in fh:
        line = line.strip()
        if line:
            provenance_records.append(json.loads(line))

print(f"Total Provenance records: {len(provenance_records)}")

# Find a MERGE record for a Condition
merge_prov = next(
    (p for p in provenance_records
     if p.get("activity", {}).get("coding", [{}])[0].get("code") == "MERGE"
     and any("Condition" in (t.get("reference","")) for t in p.get("target",[]))),
    None
)

if merge_prov:
    print()
    print("Sample MERGE Provenance:")
    print("  activity:", merge_prov["activity"]["coding"][0]["code"])
    print("  target:", [t.get("reference") for t in merge_prov.get("target", [])])
    print("  entity sources:", [e.get("what",{}).get("reference") for e in merge_prov.get("entity",[])])
    print("  agent:", [a.get("who",{}).get("display") for a in merge_prov.get("agent",[])])


## Summary

The gold tier for Rhett759 is the output of a 3-source harmonization pipeline:

| Artifact | Description | Status |
|---|---|---|
| 1 | Hyperlipidemia: SNOMED 55822004 + ICD-10 E78.5 → 1 Condition, CUI C0020473 | ✅ |
| 2 | Statin divergence: simvastatin (active) + atorvastatin (stopped) → EXT_CONFLICT_PAIR | ✅ |
| 3 | Claims: 136 Claim + 59 EoB pass-through from synthea-payer | ✅ |
| 4 | Synthesized clinical note → DocumentReference in gold; Condition extraction = Phase 2 | ⚠️ |
| 5 | Creatinine 1.4 mg/dL: epic-ehi + lab-pdf → 1 merged Observation, max quality 0.78 | ✅ |

The Streamlit console at port 8503 (`make console`) visualizes these outputs interactively.


Series complete. Start over at [00_welcome_and_setup.ipynb](./00_welcome_and_setup.ipynb) or explore the Streamlit console.